<a href="https://colab.research.google.com/github/marcinsawinski/scikit-learn-mooc/blob/main/workbooks/m6_03_ensemble_hyperparameters.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📒 Workbook M6.03 — Tuning ensembles, and asking more of them

In the lab we looked at the hyperparameters that matter for random
forests and for gradient boosting, and tuned them with a randomized
search. In this workbook you see why "more trees" does not mean the same
thing for the two families, and then you ask the football-wage model for
things a plain prediction does not give: a relationship that makes
sense, a range instead of a single number, and an honest answer to
"which features matter?".

**How this workbook works**

- The code is written for you. You do not need to write code, but you need
  to **understand every step**.
- Every `____` is a gap. Fill it in with the value asked for in the text
  **above** the cell. The error `NameError: name '____' is not defined`
  means: a gap is still empty.
- `check(...)` tells you whether your value is right (✅) or stops with a
  hint (❌).
- `commit(...)` records a **prediction** before you see the result. Be
  honest: a wrong prediction costs nothing, it is how you learn.
- Questions marked **🧠 Think** are not checked here. You will need the
  answers for the Moodle quiz and for the discussion at the start of the
  next lab.

In [ ]:
#@title Setup — run this cell first (no need to read it)
import hashlib
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

ANSWERS = {
    "cost_of_constraint": "55edad7c6cf5e696",
    "honest_measure": "0c3cb15a8f1baf1f",
    "more_trees_can_hurt": "443f4d3ed561f2f2",
    "n_fits": "b7a56873cd771f2c",
    "range_holds_on_new_players": "9390298f3fb0c5b1",
    "regime_learning_rate_1": "a1f32f577f69843e",
    "regime_smallest_learning_rate": "5a43f1e23e0f76ff"
}

def _normalize(value):
    if isinstance(value, float) and value.is_integer():
        value = int(value)
    if isinstance(value, float):
        value = round(value, 2)
    if isinstance(value, (list, tuple, set)):
        return ",".join(sorted(str(v).strip().lower() for v in value))
    return str(value).strip().lower()


def _hash(value):
    return hashlib.sha256(_normalize(value).encode()).hexdigest()[:16]


def check(name, value, hint="Re-read the text above this cell and try again."):
    """Stop with a hint unless `value` is the expected answer for `name`."""
    if _hash(value) != ANSWERS[name]:
        raise ValueError(f"❌ {name} = {value!r} is not right. {hint}")
    print(f"✅ {name}")


def check_score(name, value, actual):
    """Stop with a hint unless `value` is `actual` rounded to 2 decimals.

    For results that differ slightly between computers (e.g. k-NN ties).
    """
    if not isinstance(value, (int, float)) or abs(value - round(actual, 2)) > 0.005:
        raise ValueError(
            f"❌ {name} = {value!r} is not right. "
            "Read the score printed above and round it to 2 decimals."
        )
    print(f"✅ {name}")


def commit(prediction, options):
    """Record a prediction before seeing the result."""
    if prediction not in options:
        raise ValueError(f"Choose one of {options} first.")
    print(f"📌 Your prediction: {prediction!r}. Now run the next cell and compare.")

## 1. Data

The 8000 players of the previous workbooks (video game *FIFA 22*), 27
numerical ratings, and the weekly wage `wage_eur` to predict. Errors are
mean absolute errors in euros per week.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

players = pd.read_csv(
    "https://raw.githubusercontent.com/marcinsawinski/scikit-learn-mooc/refs/heads/main/datasets/football_players.csv"
)
numerical = [name for name in players.columns if name not in ("wage_eur", "nationality_name")]
data_train, data_test, target_train, target_test = train_test_split(
    players[numerical], players["wage_eur"], random_state=0
)

## 2. More trees: two different stories

### In a forest

A forest averages its trees. We train 150 trees once and compute the
error of the average of the first 1, 2, 3… trees.

In [ ]:
forest = RandomForestRegressor(n_estimators=150, random_state=0, n_jobs=2).fit(data_train, target_train)
test_values = data_test.to_numpy()
running_sum = np.zeros(len(data_test))
forest_curve = []
for n_trees, tree in enumerate(forest.estimators_, start=1):
    running_sum += tree.predict(test_values)
    forest_curve.append(mean_absolute_error(target_test, running_sum / n_trees))
print(f"error with 10 trees: {forest_curve[9]:.0f} €, 50 trees: {forest_curve[49]:.0f} €, 150 trees: {forest_curve[149]:.0f} €")

### In a boosting model

A boosting model adds its trees. Two settings work together:
`max_iter` (the number of trees) and `learning_rate` (the share of each
tree that is added). We draw the testing error after each tree for three
learning rates.

In [ ]:
boosting_curves = {}
for learning_rate in [1.0, 0.1, 0.003]:
    model = HistGradientBoostingRegressor(
        learning_rate=learning_rate, max_iter=300, early_stopping=False, random_state=0
    ).fit(data_train, target_train)
    boosting_curves[learning_rate] = {
        "training": [mean_absolute_error(target_train, p) for p in model.staged_predict(data_train)],
        "testing": [mean_absolute_error(target_test, p) for p in model.staged_predict(data_test)],
    }

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
axes[0].plot(range(1, 151), forest_curve, color="black")
axes[0].set(title="Random forest", xlabel="number of trees", ylabel="testing error (€ per week)", ylim=(3500, 8000))
for learning_rate, curves in boosting_curves.items():
    axes[1].plot(range(1, 301), curves["testing"], label=f"learning_rate={learning_rate}")
axes[1].set(title="Gradient boosting", xlabel="number of trees")
_ = axes[1].legend()

In [ ]:
summary = pd.DataFrame(
    {
        f"learning_rate={learning_rate}": {
            "training error, 300 trees": curves["training"][-1],
            "testing error, 300 trees": curves["testing"][-1],
            "lowest testing error": min(curves["testing"]),
            "reached with (trees)": int(np.argmin(curves["testing"])) + 1,
        }
        for learning_rate, curves in boosting_curves.items()
    }
).T
summary.round(0)

✏️ After 300 trees, name the regime of the two extreme models. Write
`"underfitting"` or `"overfitting"` for each.

- `learning_rate=1.0`: the training error is far below the testing
  error, which has been rising for a long time.
- `learning_rate=0.003`: both errors are still high and still falling.

In [ ]:
regime_learning_rate_1 = ____
regime_smallest_learning_rate = ____
check("regime_learning_rate_1", regime_learning_rate_1)
check("regime_smallest_learning_rate", regime_smallest_learning_rate)

✏️ In which family can adding trees make the model **worse**? Write
`"random forest"` or `"gradient boosting"`.

In [ ]:
more_trees_can_hurt = ____
check("more_trees_can_hurt", more_trees_can_hurt)

In a forest, the number of trees is **not a hyperparameter to tune**:
more is never worse, only slower, so take as many as you can afford. In
boosting, the number of trees and the learning rate must be chosen
**together**: a small learning rate needs many trees, a large one
overfits quickly. In practice one fixes a small learning rate and lets
early stopping (workbook M6.02) choose the number of trees.

🧠 **Think.** With `learning_rate=0.003`, somebody concludes after 300
trees that "boosting does not work on this data". What do you suggest?

## 3. A randomized search

The settings that matter most for histogram gradient boosting are the
learning rate and the size of each tree (`max_leaf_nodes`). We let a
randomized search try 8 combinations, with early stopping choosing the
number of trees each time.

✏️ 8 candidates, 3 folds, and the best one is trained again on all the
training data at the end. How many trainings in total?

In [ ]:
n_fits = ____
check("n_fits", n_fits)

In [ ]:
from scipy.stats import loguniform, randint
from sklearn.model_selection import RandomizedSearchCV

search = RandomizedSearchCV(
    HistGradientBoostingRegressor(max_iter=1000, early_stopping=True, random_state=0),
    param_distributions={"learning_rate": loguniform(0.01, 1), "max_leaf_nodes": randint(2, 64)},
    n_iter=8,
    cv=3,
    scoring="neg_mean_absolute_error",
    random_state=0,
)
search.fit(data_train, target_train)
results = pd.DataFrame(search.cv_results_)[
    ["param_learning_rate", "param_max_leaf_nodes", "mean_test_score", "std_test_score"]
].sort_values("mean_test_score", ascending=False)
results["mean_test_score"] = -results["mean_test_score"]
results.rename(columns={"mean_test_score": "mean error (€)", "std_test_score": "std"}).round(3)

🧠 **Think.** Compare the mean errors of the first few candidates with
their `std`. As in workbook M3.02: is there one best setting, or a
family of settings that are equally good? What do they have in common?

## 4. ➕ Going further: a model that respects common sense

How does the predicted wage change with the `overall` rating, all other
ratings being what they are? A **partial dependence** curve answers:
for each value of the rating, we give that rating to *every* test
player, predict, and average.

In [ ]:
def wage_against_rating(model):
    ratings = np.arange(50, 93)
    curve = []
    for rating in ratings:
        modified = data_test.assign(overall=rating)
        curve.append(model.predict(modified).mean())
    return ratings, np.array(curve)


free_model = HistGradientBoostingRegressor(random_state=0).fit(data_train, target_train)
ratings, free_curve = wage_against_rating(free_model)
steps_down = int((np.diff(free_curve) < 0).sum())
print(f"the predicted wage goes DOWN at {steps_down} of the {len(ratings) - 1} steps; largest drop: {-np.diff(free_curve).min():.0f} €")

According to the model, there are ratings at which becoming a **better**
player earns you **less**. No club works like that. The model has
picked up noise: by chance, the training players with a rating of, say,
86 earn a little less than those rated 85.

We know something the model does not: the wage should never decrease
when the rating increases. A **monotonic constraint** tells it so.

🔮 **Predict first.** Forcing the model to respect this rule will make
its testing error **"much worse"**, **"about the same"** or **"much
better"**?

In [ ]:
my_prediction = ____
commit(my_prediction, ["much worse", "about the same", "much better"])

In [ ]:
constrained_model = HistGradientBoostingRegressor(monotonic_cst={"overall": 1}, random_state=0)
constrained_model.fit(data_train, target_train)
_, constrained_curve = wage_against_rating(constrained_model)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, curve, title in [(axes[0], free_curve, "without constraint"), (axes[1], constrained_curve, "wage may not decrease with the rating")]:
    ax.plot(ratings, curve, color="black")
    ax.set(title=title, xlabel="overall rating", xlim=(75, 92))
axes[0].set_ylabel("mean predicted wage (€ per week)")

model_errors = {
    "without constraint": mean_absolute_error(target_test, free_model.predict(data_test)),
    "with constraint": mean_absolute_error(target_test, constrained_model.predict(data_test)),
}
pd.Series(model_errors, name="testing error (€)").round(0).to_frame()

✏️ What did the constraint cost in accuracy? Write `"a lot"` or `"almost
nothing"`.

In [ ]:
cost_of_constraint = ____
check("cost_of_constraint", cost_of_constraint)

For the same accuracy we get a model whose behaviour can be defended in
front of a player's agent. Knowledge of the domain entered the model as
a constraint, not as data. scikit-learn's example
[Monotonic constraints](https://scikit-learn.org/1.8/auto_examples/ensemble/plot_monotonic_constraints.html)
shows the same on artificial data.

## 5. ➕ Going further: a range instead of a number

"This player should earn €12 400" hides how unsure the model is. A club
would rather hear "between €4000 and €30 000". With `loss="quantile"`,
a boosting model predicts a **quantile** of the wage in place of its
mean: with `quantile=0.05`, a value that 5% of similar players fall
below; with `quantile=0.95`, a value that 95% fall below. Between the
two lies a range meant to contain 90% of the players.

🔮 **Predict first.** Which share of the **test** players will have
their real wage inside this range? **"more than 90%"**, **"exactly
90%"** or **"less than 90%"**?

In [ ]:
my_prediction = ____
commit(my_prediction, ["more than 90%", "exactly 90%", "less than 90%"])

In [ ]:
low = HistGradientBoostingRegressor(loss="quantile", quantile=0.05, random_state=0).fit(data_train, target_train)
high = HistGradientBoostingRegressor(loss="quantile", quantile=0.95, random_state=0).fit(data_train, target_train)


def coverage(data, target):
    return ((target >= low.predict(data)) & (target <= high.predict(data))).mean()


coverage_train, coverage_test = coverage(data_train, target_train), coverage(data_test, target_test)
print(f"share of training players inside their range: {coverage_train:.1%}")
print(f"share of test players inside their range:     {coverage_test:.1%}")

examples = data_test.head(5)
pd.DataFrame(
    {
        "overall": examples["overall"],
        "low (5%)": low.predict(examples).round(-2),
        "high (95%)": high.predict(examples).round(-2),
        "real wage": target_test.head(5),
    }
)

✏️ Does the "90% range" contain 90% of the players the model has never
seen? Write `"yes"` or `"no"`.

In [ ]:
range_holds_on_new_players = ____
check("range_holds_on_new_players", range_holds_on_new_players)

The two quantile models have overfitted a little, like any model, and
their range is too narrow for new players. A range announced as "90%"
must be **checked on held-out data**, exactly like an accuracy.

## 6. ➕ Going further: which features matter?

A forest offers `feature_importances_`: how much each feature
contributed to reducing the impurity in its trees. To test this tool we
add a column that **cannot** matter: a random number per player.

🔮 **Predict first.** Among the 28 features (27 real ones and the random
number), where will the forest's `feature_importances_` rank the random
number? **"in the top 5"**, **"in the middle"** or **"last"**?

In [ ]:
my_prediction = ____
commit(my_prediction, ["in the top 5", "in the middle", "last"])

In [ ]:
from sklearn.inspection import permutation_importance

rng = np.random.default_rng(0)
train_plus = data_train.assign(random_number=rng.normal(size=len(data_train)))
test_plus = data_test.assign(random_number=rng.normal(size=len(data_test)))
importance_forest = RandomForestRegressor(n_estimators=100, random_state=0, n_jobs=2).fit(train_plus, target_train)

permutation = permutation_importance(importance_forest, test_plus, target_test, n_repeats=5, random_state=0)
importances = pd.DataFrame(
    {
        "impurity importance": importance_forest.feature_importances_,
        "permutation importance (test set)": permutation.importances_mean,
    },
    index=train_plus.columns,
)
ranks = importances.rank(ascending=False).astype(int)
print("rank of the random number among 28 features:")
print(ranks.loc["random_number"].to_string())
importances.sort_values("impurity importance", ascending=False).head(6).round(3)

The impurity importance places a column of pure noise among the most
important features, above almost all the real ratings.

**Permutation importance** asks a different question: on the **test**
players, how much worse do the predictions get when the values of one
column are shuffled? Shuffling a column the model truly relies on hurts;
shuffling noise changes nothing.

✏️ Which of the two measures correctly says that the random number is
useless? Write `"impurity importance"` or `"permutation importance"`.

In [ ]:
honest_measure = ____
check("honest_measure", honest_measure)

Each tree of the forest is deep enough to memorise its training players,
and to do so it splits on whatever helps tell them apart. A random
number with thousands of different values is perfect for that. The
impurity importance is computed **on the training data** and counts
those splits as useful. It favours features with many different values,
whether they carry information or not.

scikit-learn's example
[Permutation importance vs random forest feature importance](https://scikit-learn.org/1.8/auto_examples/inspection/plot_permutation_importance.html)
shows the same on the passengers of the Titanic.

🧠 **Think.**

1. `overall` dominates both rankings. It is a summary of the other
   ratings, computed by the game. If you removed it, would the other
   ratings become more important? What does that say about reading an
   importance as "the effect of this feature"?
2. Adding a random column is a cheap test you can run on any model.
   What would you conclude about real features that rank **below** it?

## 🎤 Before the next lab

1. Do the **Moodle quiz** for this workbook (mandatory, not graded).
2. At the start of the next lab, some of you will be asked to present
   (3 minutes, using your notebook on the projector) **one** of:
   - more trees in a forest and in a boosting model: the two pictures;
   - learning rate and number of trees: the three curves and their
     regimes;
   - the monotonic constraint: what was wrong without it and what it
     cost;
   - the 90% range that does not contain 90% of the new players;
   - the random number that came out as an important feature.